# GenCare AI finetuning
---
Auteur:   Eva Rombouts  
Datum:    26-12-2023.  
Doel:     Ervaring opdoen met Python, NLP  
Doel project: Finetuning van model  

Data gegenereerd met de OpenAI API.  

---

In [2]:
#from datasets import load_dataset, Dataset, DatasetDict
#from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
#import torch
#import time
#import evaluate
import pandas as pd
import numpy as np
#from peft import LoraConfig, get_peft_model, TaskType
#from sklearn.model_selection import train_test_split

In [3]:
# # Wanneer in colab
# from google.colab import drive
# drive.mount('/content/drive')
# gci = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/gci_rapportages.csv')
# gci.info()

In [5]:
gci = pd.read_csv('../zorgdata/gci_rapportages.csv')
gci.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1402 entries, 0 to 1401
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliënt_id    1402 non-null   object 
 1   week         1402 non-null   int64  
 2   dag          1402 non-null   object 
 3   niveau       1402 non-null   object 
 4   rapportage   1402 non-null   object 
 5   onrustscore  1395 non-null   float64
dtypes: float64(1), int64(1), object(4)
memory usage: 65.8+ KB


In [11]:
gci['onrustscore'].describe()

count    1395.000000
mean       44.365591
std        20.635380
min         0.000000
25%        30.000000
50%        40.000000
75%        60.000000
max        95.000000
Name: onrustscore, dtype: float64

In [6]:
gci['rapportage'].head()

0    Meneer Pietersen heeft vanochtend moeite gehad...
1    Vanochtend was meneer Pietersen erg onrustig e...
2    Gisteravond had meneer Pietersen moeite met sl...
3    Vandaag had meneer Pietersen veel pijn in zijn...
4    Meneer Pietersen was vanochtend erg onrustig e...
Name: rapportage, dtype: object

In [15]:
# Om een gevoel te krijgen van de lengte van de rapportages:
gci['rapportage_lengte'] = gci['rapportage'].apply(len)
gci['n_woorden'] = gci['rapportage'].str.split().apply(len)
gci[['rapportage_lengte','n_woorden']].describe()


,rapportage_lengte,n_woorden
count,1402.000000,1402.000000
mean,578.138374,98.608417
std,136.105559,24.054038
min,293.000000,49.000000
25%,478.000000,81.000000
50%,555.000000,94.000000
75%,655.750000,112.750000
max,1120.000000,190.000000


## Modellen zoeken

In [7]:
from transformers import pipeline

In [8]:
model = 'wietsedv/bert-base-dutch-cased-finetuned-sentiment'
sentiment_analysis = pipeline('sentiment-analysis', model = model)

In [32]:
# Laten we starten met de eerste week van dhr Pieterse
test_zinnen = gci.loc[(gci['cliënt_id'] == 'kamer02') & (gci['week'] == 1), 'rapportage'].tolist()
for rp in test_zinnen:
    print(rp)

Sophie had vanochtend wat moeite met opstaan uit bed. Haar mobiliteitsproblemen leken verergerd en ze had wat ondersteuning nodig om uit bed te komen. Na het ontbijt hebben we samen een wandeling gemaakt door de tuin. Sophie genoot er zichtbaar van en kon wat kleine taken uitvoeren, zoals bloemen plukken. In de middag hebben we cognitieve stimulatieactiviteiten gedaan, zoals herinneringskaarten en puzzels. Sophie uitte af en toe frustratie omdat ze moeite had om zich bepaalde dingen te herinneren, maar haar gezicht opklaarde zodra ze een puzzelstuk op de juiste plek kon leggen. Ze krijgt regelmatig haar medicatie, en haar lichamelijke klachten lijken vandaag meer invloed te hebben op haar algemene welzijn.
Vanochtend had Sophie wat moeite met het eten van haar ontbijt. Ze had wat hulp nodig bij het snijden van haar brood en het smeren van de boter. Haar handschrift was ook moeilijk te lezen toen ze een brief wilde schrijven naar haar dochter. In de middag hebben we samen wat lichte oef

In [33]:
se_output = sentiment_analysis(test_zinnen)

for a in enumerate(se_output) :
    print(a)

(0, {'label': 'pos', 'score': 0.9968289732933044})
(1, {'label': 'neg', 'score': 0.9953723549842834})
(2, {'label': 'neg', 'score': 0.9999483823776245})
(3, {'label': 'pos', 'score': 0.9999709129333496})
(4, {'label': 'pos', 'score': 0.9999963045120239})


Ik ben niet erg onder de indruk... Hij neigt erg naar positiviteit. 
Laten we er nog een proberen

In [34]:
model="henryk/bert-base-multilingual-cased-finetuned-dutch-squad2"

qa_pipeline = pipeline(
    "question-answering",
    model = model,
    tokenizer = model
)

Some weights of the model checkpoint at henryk/bert-base-multilingual-cased-finetuned-dutch-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [37]:
for rp in test_zinnen:
    qa_resultaat = qa_pipeline({
        'context': rp,
        'question': "Hoe heet deze client?"})
#        'question': "Wat heeft de client nodig?"})
#        'question': "Hoe voelt deze client zich?"})
    print(rp)
    print(qa_resultaat['answer'])
    print('-'*100)


Sophie had vanochtend wat moeite met opstaan uit bed. Haar mobiliteitsproblemen leken verergerd en ze had wat ondersteuning nodig om uit bed te komen. Na het ontbijt hebben we samen een wandeling gemaakt door de tuin. Sophie genoot er zichtbaar van en kon wat kleine taken uitvoeren, zoals bloemen plukken. In de middag hebben we cognitieve stimulatieactiviteiten gedaan, zoals herinneringskaarten en puzzels. Sophie uitte af en toe frustratie omdat ze moeite had om zich bepaalde dingen te herinneren, maar haar gezicht opklaarde zodra ze een puzzelstuk op de juiste plek kon leggen. Ze krijgt regelmatig haar medicatie, en haar lichamelijke klachten lijken vandaag meer invloed te hebben op haar algemene welzijn.
Sophie
----------------------------------------------------------------------------------------------------
Vanochtend had Sophie wat moeite met het eten van haar ontbijt. Ze had wat hulp nodig bij het snijden van haar brood en het smeren van de boter. Haar handschrift was ook moeili

In [38]:
sum_model = 'yhavinga/t5-v1.1-base-dutch-cnn-test'

In [39]:
# Maak de summarization pipeline
summarization_pipeline = pipeline(
    "summarization",
    model=sum_model,
    tokenizer=sum_model
)

In [ ]:
# Voer de samenvatting uit
samenvatting_resultaat = summarization_pipeline(test_zinnen[1])

In [42]:
samenvatting = samenvatting_resultaat[0]['summary_text']
print(samenvatting)

Sophie had wat hulp nodig bij het snijden van haar brood en het smeren van de boter. Ze had ook moeite met het schrijven van een brief naar haar dochter. In de middag hebben we samen wat muziektherapie gedaan om haar mobiliteit te verbeteren. Ze is nu gemotiveerd, maar lijkt over het algemeen tevreden .


In [46]:
for rp in test_zinnen:
    sum_resultaat = summarization_pipeline(rp, min_length = 50, max_length=70)
    smry = sum_resultaat[0]['summary_text']
    print(rp)
    print(smry)
    print('-'*100)


Sophie had vanochtend wat moeite met opstaan uit bed. Haar mobiliteitsproblemen leken verergerd en ze had wat ondersteuning nodig om uit bed te komen. Na het ontbijt hebben we samen een wandeling gemaakt door de tuin. Sophie genoot er zichtbaar van en kon wat kleine taken uitvoeren, zoals bloemen plukken. In de middag hebben we cognitieve stimulatieactiviteiten gedaan, zoals herinneringskaarten en puzzels. Sophie uitte af en toe frustratie omdat ze moeite had om zich bepaalde dingen te herinneren, maar haar gezicht opklaarde zodra ze een puzzelstuk op de juiste plek kon leggen. Ze krijgt regelmatig haar medicatie, en haar lichamelijke klachten lijken vandaag meer invloed te hebben op haar algemene welzijn.
Sophie's mobiliteitsproblemen lijken verergerd te zijn. Ze heeft cognitieve stimulatieactiviteiten gedaan, zoals herinneringskaarten en puzzels. Ze krijgt regelmatig haar medicatie, e
----------------------------------------------------------------------------------------------------

Wow, dat werkt best goed. 

Maar nu naar mijn taak. Onrustscores voorspellen... ChatGPT geeft de volgende stappoen:


Voor het voorspellen van een onrustscore, die in essentie een vorm van regressie is (waarbij het doel is om een continue waarde te voorspellen), zou u een getransformeerd taalmodel kunnen gebruiken dat is aangepast voor regressietaken. Hier zijn een paar stappen en overwegingen:

Modelkeuze: Begin met een voorgeleerd taalmodel zoals BERT, RoBERTa, of DistilBERT. Deze modellen zijn al getraind op grote tekstcorpora en hebben een goed begrip van taalstructuren.

Aanpassen voor Regressie: Hoewel deze modellen doorgaans worden gebruikt voor classificatie of andere NLP-taken, kunnen ze worden aangepast voor regressie. Dit houdt meestal in dat de laatste laag (vaak een classificatielaag) wordt vervangen door een regressielaag (zoals een lineaire laag die één output produceert).

Training: U moet het model trainen met uw specifieke dataset. Dit betekent het voeden van de teksten aan het model en het laten leren van de overeenkomstige onrustscores. Dit proces omvat het definiëren van een geschikte loss-functie voor regressie, zoals Mean Squared Error (MSE) of Mean Absolute Error (MAE).

Gegevensvoorbereiding: Zorg ervoor dat uw gegevens goed zijn voorbereid en geformatteerd. Dit betekent het tokeniseren van de teksten op een manier die compatibel is met het gekozen model en het normaliseren van uw onrustscores (indien nodig).

Evaluatie: Beoordeel de prestaties van uw model met behulp van geschikte regressiemetrieken, zoals MSE, MAE, of R-squared.

Fijnafstelling en Optimalisatie: Mogelijk moet u het model finetunen en de hyperparameters optimaliseren om de beste prestaties voor uw specifieke toepassing te bereiken.

Gebruik van een Domeinspecifiek Model: Als er modellen beschikbaar zijn die specifiek zijn getraind op medische of zorggerelateerde gegevens, kunnen deze modellen beter presteren vanwege hun domeinspecifieke kennis.

Een concrete implementatie zou kunnen beginnen met een model als bert-base-uncased (of een Nederlandstalige variant indien uw teksten in het Nederlands zijn), waarbij u vervolgens een regressielaag toevoegt en het model traint met uw dataset van zorgrapportages en bijbehorende onrustscores.

Let's go...  
Stap 1  

Voor het voorspellen van een onrustscore, die in essentie een vorm van regressie is (waarbij het doel is om een continue waarde te voorspellen), zou u een getransformeerd taalmodel kunnen gebruiken dat is aangepast voor regressietaken. Hier zijn een paar stappen en overwegingen:

Modelkeuze: Begin met een voorgeleerd taalmodel zoals BERT, RoBERTa, of DistilBERT. Deze modellen zijn al getraind op grote tekstcorpora en hebben een goed begrip van taalstructuren.


In [47]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

tokenizer = BertTokenizer.from_pretrained('wietsedv/bert-base-dutch-cased')
model = BertForSequenceClassification.from_pretrained('wietsedv/bert-base-dutch-cased')


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at wietsedv/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.weight', 'classifier.weight', 'bert.pooler.dense.bias', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [48]:
from transformers import BertTokenizer, BertModel
import torch.nn as nn

tokenizer = BertTokenizer.from_pretrained('wietsedv/bert-base-dutch-cased')
model = BertModel.from_pretrained('wietsedv/bert-base-dutch-cased')

Some weights of BertModel were not initialized from the model checkpoint at wietsedv/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [49]:
class BertForRegression(nn.Module):
    def __init__(self, bert_model):
        super(BertForRegression, self).__init__()
        self.bert = bert_model
        # De regressielaag: BERT's outputgrootte (768 voor base-modellen) naar 1
        self.regression = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        # De output van BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Neem de output van de laatste verborgen laag voor de [CLS] token
        cls_output = outputs.last_hidden_state[:, 0, :]
        # Pas de regressielaag toe
        return self.regression(cls_output)

# Maak een instantie van het aangepaste model
regression_model = BertForRegression(model)


In [51]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch

class RapportageDataset(Dataset):
    def __init__(self, teksten, scores, tokenizer, max_len):
        self.teksten = teksten
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.teksten)

    def __getitem__(self, item):
        tekst = str(self.teksten[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            tekst,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': tekst,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }

# Parameters
max_len = 512  # Max lengte van een tekst voor BERT
batch_size = 16
learning_rate = 2e-5
epochs = 3

# Dataset splitsen
df_train, df_val = train_test_split(gci, test_size=0.1)

# Datasets voorbereiden
train_data = RapportageDataset(
    teksten=df_train['rapportage'].to_numpy(),
    scores=df_train['onrustscore'].to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
)

val_data = RapportageDataset(
    teksten=df_val['rapportage'].to_numpy(),
    scores=df_val['onrustscore'].to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)


In [52]:
# Duurt een uur
from transformers import AdamW
from tqdm import tqdm

# Zet het model en optimizer op
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
regression_model.to(device)
optimizer = AdamW(regression_model.parameters(), lr=learning_rate)

# Loss functie
loss_fn = nn.MSELoss()

# Trainingsloop
for epoch in range(epochs):
    regression_model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        outputs = regression_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, scores.unsqueeze(1))
        
        loss.backward()
        optimizer.step()

        loop.set_description(f'Epoch {epoch}')
        loop.set_postfix(loss=loss.item())


/Users/eva/anaconda3/envs/nlp_env/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 2: 100%|██████████| 79/79 [18:31<00:00, 14.06s/it, loss=nan]


In [58]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def evaluate_model(model, data_loader, device):
    model.eval()
    voorspellingen = []
    werkelijke_scores = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            voorspellingen.extend(outputs.flatten().cpu().numpy())
            werkelijke_scores.extend(scores.flatten().cpu().numpy())

    mse = mean_squared_error(werkelijke_scores, voorspellingen)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(werkelijke_scores, voorspellingen)

    return mse, rmse, mae


In [62]:
type(val_data)

__main__.RapportageDataset

In [64]:
df_val.dropna(inplace=True)
val_data = RapportageDataset(
    teksten=df_val['rapportage'].to_numpy(),
    scores=df_val['onrustscore'].to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
)


In [65]:
# Zorg ervoor dat uw testset is voorbereid op dezelfde manier als uw trainings- en validatiesets
test_loader = DataLoader(val_data, batch_size=batch_size)

In [66]:
type(test_loader)

torch.utils.data.dataloader.DataLoader

In [67]:
# Model Evaluatie
mse, rmse, mae = evaluate_model(regression_model, test_loader, device)

ValueError: Input contains NaN.

In [ ]:
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')


In [ ]:
# Zorg ervoor dat uw testset is voorbereid op dezelfde manier als uw trainings- en validatiesets
test_loader = DataLoader(val_data, batch_size=batch_size)

In [ ]:
type(test_loader)

torch.utils.data.dataloader.DataLoader